# Automata-02 — Expressions régulières : le théorème de Kleene, dans les deux sens

**Navigation** : [Index de la série](README.md) | [Automata-01](Automata-01-AutomatesFinis-CSharp.ipynb) | [Version Python](Automata-02-ExpressionsRegulieres-Python.ipynb)

> **Public : Découverte.** Deuxième carnet de la sous-série. Il reprend l'AFN à
> epsilon-transitions d'`Automata-01`, n'ajoute aucune dépendance (bibliothèque de base
> .NET seule) et se sert de `Regex` comme **oracle externe**.

## Ce que ce notebook suppose

- savoir lire un AFD, un AFN, et ce qu'est une epsilon-transition (`Automata-01`, sections 1 à 3) ;
- C# : enregistrements (`record`), collections génériques, LINQ, fonctions locales.

## Ce qu'on y apprend

1. **Décrire** un langage par une expression régulière, puis convertir
   **expression → automate** par induction structurelle sur la syntaxe.
2. **Convertir** dans l'autre sens, **automate → expression**, en éliminant les états.
3. **Mesurer** ce que coûte chaque conversion, au lieu de l'affirmer.

**Vérification indépendante.** Les deux convertisseurs de ce carnet sont écrits ici, donc
ils pourraient se tromper ensemble. Chaque résultat est donc confronté au moteur de regex
de la bibliothèque de base (`Regex.IsMatch`, ancré par `\A(?:…)\z`), qui est une
**implémentation séparée**, sur **tous** les mots jusqu'à une longueur fixée.


## 1. Deux notations pour la même famille de langages

Une **expression régulière** se construit à partir de trois briques de base — l'ensemble
vide $\varnothing$, le mot vide $\varepsilon$, et une lettre $a$ de l'alphabet — et de
trois opérations : l'**union** $r \mid s$, la **concaténation** $rs$, et l'**étoile** $r^\star$.

Les trois opérations sont ordonnées par priorité : l'étoile lie le plus fort, la
concaténation ensuite, l'union le plus faible. C'est pour cela que `(a|b)*abb` se lit
« n'importe quel mot sur $\{a, b\}$, puis `abb` » et non « `a`, ou `b` répété, puis `abb` ».

Le manuel d'*Introduction to Automata Theory, Languages, and Computation*
(Hopcroft, Motwani & Ullman, 2001 ; copie archivée, chapitre 3) démontre l'équivalence
entre les deux notations en **deux théorèmes séparés** :

| Énoncé du manuel | Sens | Où |
|---|---|---|
| **Théorème 3.4** : si $L = L(A)$ pour un AFD $A$, il existe une expression régulière $R$ telle que $L = L(R)$ | automate → expression | p. 91, §3.2.1 |
| **Théorème 3.7** : tout langage défini par une expression régulière est défini par un automate fini | expression → automate | p. 102, §3.2.3 |

C'est leur **réunion** qu'on appelle le théorème de Kleene. Le manuel la démontre par
deux constructions distinctes, et ce carnet les implémente toutes les deux : la
section 2 suit le théorème 3.7, la section 3 suit le §3.2.2 (variante par élimination
d'états du théorème 3.4).

Pour manipuler les expressions de façon uniforme, on les représente comme un **arbre**
dont les feuilles sont $\varnothing$, $\varepsilon$ ou une lettre, et dont les nœuds
internes portent une opération. C'est exactement la structure sur laquelle l'induction
du théorème 3.7 travaille.


In [1]:
// Une expression reguliere est un ARBRE : un enregistrement (record) par sorte de noeud.
// Les records se comparent par valeur, comme les tuples du jumeau Python : les operateurs
// restent composables et la taille de l'arbre mesurable, sans dependance externe.
#nullable enable

abstract record Rx;                      // toute expression reguliere
sealed record Vide : Rx;                 // langage vide : aucun mot du tout
sealed record Eps : Rx;                  // epsilon : le langage reduit au mot vide
sealed record Lit(char Lettre) : Rx;     // le langage {lettre}
sealed record Union(Rx X, Rx Y) : Rx;    // L(X) union L(Y)
sealed record Concat(Rx X, Rx Y) : Rx;   // {u + v : u dans L(X), v dans L(Y)}
sealed record Etoile(Rx X) : Rx;         // L(X)*, fermeture de Kleene

// Parenthese un sous-arbre quand son operateur est plus faible que celui du contexte.
string Entre(Rx? rx, char contexte) {
    if (rx is null) return "(?!)";
    if (rx is Union && contexte != 'u') return $"({Rendre(rx)})";
    if (rx is Concat && contexte == 's') return $"({Rendre(rx)})";
    return Rendre(rx);
}

// Rendu de l'arbre en chaine lisible par Regex : parentheses seulement ou il en faut.
string Rendre(Rx? rx) {
    if (rx is null) return "(?!)";       // assertion toujours fausse : ne matche aucun mot
    switch (rx) {
        case Vide: return "(?!)";
        case Eps: return "";             // epsilon s'ecrit comme la chaine vide
        case Lit l: return l.Lettre.ToString();
        case Union u: return $"{Entre(u.X, 'u')}|{Entre(u.Y, 'u')}";
        case Concat c: return $"{Entre(c.X, 'c')}{Entre(c.Y, 'c')}";
    }
    return $"{Entre(((Etoile)rx).X, 's')}*";   // dernier cas : l'etoile
}

// Nombre de noeuds de l'arbre : la mesure qui predit la taille de l'automate de Thompson.
int NbNoeuds(Rx? rx) => rx switch {
    null => 1,
    Vide or Eps or Lit => 1,
    Etoile s => 1 + NbNoeuds(s.X),
    Union u => 1 + NbNoeuds(u.X) + NbNoeuds(u.Y),
    _ => 1 + NbNoeuds(((Concat)rx).X) + NbNoeuds(((Concat)rx).Y),
};

// Trois expressions de reference, ecrites comme des arbres (et non comme des chaines) :
// AB designe le fragment a|b, reutilise dans les constructions qui suivent.
var AB = new Union(new Lit('a'), new Lit('b'));
var rxAbb = new Concat(new Concat(new Concat(new Etoile(AB), new Lit('a')), new Lit('b')), new Lit('b'));
var rxAbc = new Concat(new Lit('a'), new Etoile(new Union(new Lit('b'), new Lit('c'))));
var rxAbEtoile = new Etoile(new Concat(new Lit('a'), new Lit('b')));

foreach (var (nom, rx) in new (string, Rx)[] {
    ("(a|b)*abb", rxAbb), ("a(b|c)*", rxAbc), ("(ab)*", rxAbEtoile),
})
    Console.WriteLine($"{nom,-10} noeuds={NbNoeuds(rx),2}  rendu='{Rendre(rx)}'");

The below script needs to be able to find the current output cell; this is an easy method to get it.

(a|b)*abb  noeuds=10  rendu='(a|b)*abb'


a(b|c)*    noeuds= 6  rendu='a(b|c)*'


(ab)*      noeuds= 4  rendu='(ab)*'


**Lecture.** Les trois arbres se rendent exactement comme leurs notations usuelles :
`(a|b)*abb`, `a(b|c)*`, `(ab)*` — dix, six et quatre nœuds. Les parenthèses n'apparaissent
que là où la priorité l'exige, ce qui compte pour la suite : c'est cette même chaîne que le
moteur externe lira, et un rendu faux ferait diverger l'oracle sans que nos automates
soient en cause.

Le compteur de nœuds est là pour une raison précise : la construction de la section
suivante crée **deux états par nœud**, donc il annonce la taille de l'automate avant
même de le construire.


## 2. De l'expression à l'automate : induction structurelle

Le théorème 3.7 se démontre par **induction structurelle** sur l'arbre, avec trois cas de
base et trois cas d'induction. L'énoncé du manuel impose trois invariants au fragment
construit pour chaque sous-expression : un **seul** état final, aucune flèche qui entre
par l'état initial, aucune flèche qui sort de l'état final. Ce sont eux qui rendent la
composition possible sans conflit.

| Cas | Fragment | États créés |
|---|---|---|
| base $\varnothing$ | entrée et sortie isolées, aucun chemin | 2 |
| base $\varepsilon$ | une epsilon-flèche entrée → sortie | 2 |
| base $a$ | une flèche étiquetée $a$ | 2 |
| $r \mid s$ | nouvelle entrée branchée sur les deux fragments, nouvelles flèches vers la nouvelle sortie | 2 |
| $rs$ | les deux fragments en série, epsilon-flèche de la sortie du premier vers l'entrée du second | 2 |
| $r^\star$ | boucle epsilon de la sortie vers l'entrée, plus un saut epsilon entrée → sortie | 2 |

Chaque nœud consomme donc exactement deux états neufs : le nombre d'états est
$2 \times$ (nombre de nœuds). Les epsilon-transitions ne sont pas un détail
d'implémentation ici — c'est la construction elle-même qui les produit.


In [2]:
// Un AFN a epsilon-transitions : Delta compose ((etat, lettre) -> ensemble d'etats),
// Epsilon (etat -> etats atteignables sans lire), un etat initial, des etats finaux.
#nullable enable

class AfN {
    public HashSet<int> Etats = new();
    public Dictionary<(int, char), HashSet<int>> Delta = new();
    public Dictionary<int, HashSet<int>> Epsilon = new();
    public HashSet<char> Alphabet = new();
    public int Initial;
    public HashSet<int> Finals = new();

    // Etats atteignables depuis S par epsilon-transitions seules.
    public HashSet<int> Fermeture(IEnumerable<int> S) {
        var vus = new HashSet<int>(S);
        var pile = new Stack<int>(vus);
        while (pile.Count > 0) {
            int q = pile.Pop();
            if (Epsilon.TryGetValue(q, out var cibles))
                foreach (int r in cibles)
                    if (vus.Add(r)) pile.Push(r);
        }
        return vus;
    }

    // True ssi un chemin de lecture du mot finit dans un etat final.
    public bool Accepte(string mot) {
        var courant = Fermeture(new[] { Initial });
        foreach (char lettre in mot) {
            var suivant = new HashSet<int>();
            foreach (int q in courant)
                if (Delta.TryGetValue((q, lettre), out var cibles)) suivant.UnionWith(cibles);
            courant = Fermeture(suivant);
            if (courant.Count == 0) return false;
        }
        return courant.Overlaps(Finals);
    }
}

class Thompson {
    // Expression -> epsilon-AFN, par induction structurelle (theoreme 3.7 du manuel).
    // Chaque noeud de l'arbre cree deux etats neufs : une entree et une sortie. Le fragment
    // d'un noeud a une seule entree, une seule sortie, aucune fleche entrante sur son
    // entree ni sortante de sa sortie : les trois invariants de la preuve, verifies plus bas.
    public Dictionary<(int, char), HashSet<int>> Delta = new();   // (etat, lettre) -> cibles
    public Dictionary<int, HashSet<int>> Epsilon = new();         // etat -> cibles sans lire
    int suivant = 0;                                              // compteur d'etats crees

    // Un etat neuf, numerote dans l'ordre de creation.
    int Neuf() => suivant++;

    // Ajoute une fleche, en accumulant les cibles quand la cle existe deja.
    static void Fleche(Dictionary<int, HashSet<int>> table, int cle, int cible) {
        if (!table.TryGetValue(cle, out var cibles)) table[cle] = cibles = new HashSet<int>();
        cibles.Add(cible);
    }
    static void Fleche(Dictionary<(int, char), HashSet<int>> table, (int, char) cle, int cible) {
        if (!table.TryGetValue(cle, out var cibles)) table[cle] = cibles = new HashSet<int>();
        cibles.Add(cible);
    }

    // (entree, sortie) du fragment qui reconnait exactement L(rx).
    public (int Entree, int Sortie) Fragment(Rx? rx) {
        if (rx is null) return (Neuf(), Neuf());        // base : langage vide, aucun chemin
        if (rx is Eps) {                                // base : epsilon, une fleche vide
            var (e0, s0) = (Neuf(), Neuf());
            Fleche(Epsilon, e0, s0);
            return (e0, s0);
        }
        if (rx is Lit l) {                              // base : une lettre, une fleche etiquetee
            var (e0, s0) = (Neuf(), Neuf());
            Fleche(Delta, (e0, l.Lettre), s0);
            return (e0, s0);
        }
        var (entree, sortie) = (Neuf(), Neuf());
        switch (rx) {
            case Union u:                               // induction : deux branches en parallele
                foreach (var branche in new[] { u.X, u.Y }) {
                    var (a, b) = Fragment(branche);
                    Fleche(Epsilon, entree, a);
                    Fleche(Epsilon, b, sortie);
                }
                break;
            case Concat c: {                            // induction : deux fragments en serie
                var (a, b) = Fragment(c.X);
                var (c1, d1) = Fragment(c.Y);
                Fleche(Epsilon, entree, a);
                Fleche(Epsilon, b, c1);
                Fleche(Epsilon, d1, sortie);
                break;
            }
            default: {                                  // induction : etoile, boucle + saut direct
                var (a, b) = Fragment(((Etoile)rx).X);
                Fleche(Epsilon, entree, a);
                Fleche(Epsilon, b, a);
                Fleche(Epsilon, b, sortie);
                Fleche(Epsilon, entree, sortie);
                break;
            }
        }
        return (entree, sortie);
    }

    // Alphabet effectif de l'expression : les lettres qui y apparaissent.
    static void Collecter(Rx? rx, HashSet<char> lettres) {
        switch (rx) {
            case Lit l: lettres.Add(l.Lettre); break;
            case Etoile s: Collecter(s.X, lettres); break;
            case Union u: Collecter(u.X, lettres); Collecter(u.Y, lettres); break;
            case Concat c: Collecter(c.X, lettres); Collecter(c.Y, lettres); break;
        }
    }
    public static HashSet<char> Lettres(Rx? rx) {
        var lettres = new HashSet<char>();
        Collecter(rx, lettres);
        return lettres;
    }

    // AFN complet, reconnu par l'arbre.
    public AfN Construire(Rx? rx) {
        var (initial, sortie) = Fragment(rx);
        var afn = new AfN {
            Alphabet = Lettres(rx),
            Initial = initial,
            Etats = Enumerable.Range(0, suivant).ToHashSet(),
            Finals = new HashSet<int> { sortie },
        };
        afn.Delta = Delta;
        afn.Epsilon = Epsilon;
        return afn;
    }
}

// Nombre de fleches qui arrivent sur cet etat (etiquetees ou epsilon).
int Entrantes(AfN afn, int etat) =>
    afn.Delta.Values.Count(c => c.Contains(etat)) + afn.Epsilon.Values.Count(c => c.Contains(etat));

// Nombre de fleches qui partent de cet etat (etiquetees ou epsilon).
int Sortantes(AfN afn, int etat) =>
    afn.Delta.Keys.Count(cle => cle.Item1 == etat)
    + (afn.Epsilon.TryGetValue(etat, out var cibles) ? cibles.Count : 0);

var afnAbb = new Thompson().Construire(rxAbb);
int finalAbb = afnAbb.Finals.Single();

// Les trois invariants du theoreme 3.7, mesures sur l'automate construit.
Console.WriteLine($"(a|b)*abb : {afnAbb.Etats.Count} etats pour {NbNoeuds(rxAbb)} noeuds -> deux par noeud : {afnAbb.Etats.Count == 2 * NbNoeuds(rxAbb)}");
Console.WriteLine($"un seul etat final : {afnAbb.Finals.Count == 1} ; fleches entrantes sur l'initial : {Entrantes(afnAbb, afnAbb.Initial)} ; fleches sortantes du final : {Sortantes(afnAbb, finalAbb)}");
Console.WriteLine($"arcs etiquetes : {afnAbb.Delta.Values.Sum(v => v.Count)} ; epsilon-arcs : {afnAbb.Epsilon.Values.Sum(v => v.Count)}");

// Quelques mots, pour voir la machine tourner avant de la verifier exhaustivement.
foreach (string mot in new[] { "", "abb", "aabb", "ab", "babb" }) {
    string texte = $"'{mot}'";
    Console.WriteLine($"  {texte,-8} accepte={afnAbb.Accepte(mot)}");
}

(a|b)*abb : 20 etats pour 10 noeuds -> deux par noeud : True


un seul etat final : True ; fleches entrantes sur l'initial : 0 ; fleches sortantes du final : 0


arcs etiquetes : 5 ; epsilon-arcs : 17


  ''       accepte=False


  'abb'    accepte=True


  'aabb'   accepte=True


  'ab'     accepte=False


  'babb'   accepte=True


**Lecture.** Vingt états pour dix nœuds, et les trois invariants tiennent : un seul état
final, aucune flèche entrante sur l'initial, aucune sortante du final. Les dix-sept flèches
vides se répartissent comme la construction le prévoit — quatre pour l'étoile, quatre pour
l'union (deux par branche), trois par concaténation — contre cinq arcs étiquetés, un par
lettre de l'expression. L'automate est donc surtout fait d'epsilon-transitions.

Vingt états pour un langage qu'un AFD à quatre états suffit à reconnaître : la construction
de Thompson est **simple à démontrer**, pas économique. C'est le prix de l'induction
structurelle, et la section 4 le mesure.

`"babb"` est accepté comme `"abb"` : l'étoile initiale absorbe le préfixe. `"ab"` ne l'est
pas. Rien de surprenant — mais cinq mots ne prouvent rien, d'où la vérification qui suit.


In [3]:
#nullable enable
using System.Text.RegularExpressions;

// Cle canonique d'un ensemble d'etats : triee, jointe par des virgules.
string Cle(IEnumerable<int> S) => string.Join(",", S.OrderBy(x => x));

// Un AFD a etats nommes par des chaines ; Delta[(etat, lettre)] = etat suivant (total).
class Afd {
    public HashSet<string> Etats = new();
    public HashSet<char> Alphabet = new();
    public Dictionary<(string, char), string> Delta = new();
    public string Initial = "";
    public HashSet<string> Finals = new();
}

// Construction des sous-ensembles ; un etat de l'AFD est nomme par sa cle triee.
// Nommer les etats par une chaine canonique (et non par un ensemble) rend l'automate
// lisible, et rend surtout l'ordre d'elimination de la section 3 reproductible.
Afd Determiniser(AfN afn) {
    var depart = afn.Fermeture(new[] { afn.Initial });
    var nom = new Dictionary<string, HashSet<int>> { [Cle(depart)] = depart };
    var delta = new Dictionary<(string, char), string>();
    var aTraiter = new Stack<string>();
    aTraiter.Push(Cle(depart));
    while (aTraiter.Count > 0) {
        string courant = aTraiter.Pop();
        foreach (char lettre in afn.Alphabet.OrderBy(c => c)) {
            var suivant = new HashSet<int>();
            foreach (int q in nom[courant])
                if (afn.Delta.TryGetValue((q, lettre), out var cibles)) suivant.UnionWith(cibles);
            var cible = afn.Fermeture(suivant);
            string cle = Cle(cible);
            if (!nom.ContainsKey(cle)) { nom[cle] = cible; aTraiter.Push(cle); }
            delta[(courant, lettre)] = cle;
        }
    }
    return new Afd {
        Etats = nom.Keys.ToHashSet(),
        Alphabet = afn.Alphabet,
        Delta = delta,
        Initial = Cle(depart),
        Finals = nom.Where(kv => kv.Value.Overlaps(afn.Finals)).Select(kv => kv.Key).ToHashSet(),
    };
}

// Lecture deterministe : une seule transition par lettre.
bool AfdAccepte(Afd afd, string mot) {
    string courant = afd.Initial;
    foreach (char lettre in mot) courant = afd.Delta[(courant, lettre)];
    return afd.Finals.Contains(courant);
}

// Tous les mots de longueur 0 a nMax sur l'alphabet, mot vide compris.
IEnumerable<string> Mots(char[] alphabet, int nMax) {
    yield return "";
    for (int n = 1; n <= nMax; n++)
        foreach (int[] idx in Produits(alphabet.Length, n))
            yield return new string(idx.Select(i => alphabet[i]).ToArray());
}
IEnumerable<int[]> Produits(int k, int n) {
    int[] cur = new int[n];
    while (true) {
        yield return cur.ToArray();
        int p = n - 1;
        while (p >= 0 && cur[p] == k - 1) { cur[p] = 0; p--; }
        if (p < 0) break;
        cur[p]++;
    }
}

// L'AFN a quatre etats du carnet precedent, pour le meme langage : les mots finissant par abb.
var afnQuatre = new AfN {
    Alphabet = new HashSet<char> { 'a', 'b' },
    Initial = 0,
    Etats = new HashSet<int> { 0, 1, 2, 3 },
    Finals = new HashSet<int> { 3 },
};
afnQuatre.Delta[(0, 'a')] = new HashSet<int> { 0, 1 };
afnQuatre.Delta[(0, 'b')] = new HashSet<int> { 0 };
afnQuatre.Delta[(1, 'b')] = new HashSet<int> { 2 };
afnQuatre.Delta[(2, 'b')] = new HashSet<int> { 3 };

// Une seconde expression, de forme differente : etoile des deux cotes du motif.
var rxAba = new Concat(new Etoile(AB), new Concat(new Lit('a'), new Concat(new Lit('b'), new Concat(new Lit('a'), new Etoile(AB)))));

// Oracle externe : le moteur d'expressions regulieres de la bibliotheque de base.
// \A et \z entourent le motif pour exiger, comme le fullmatch du jumeau Python, le mot
// ENTIER : sans eux, Regex.IsMatch trouverait le motif n'importe ou dans le mot.
bool Oracle(string motif, string mot) => Regex.IsMatch(mot, $@"\A(?:{motif})\z");

var tous = Mots(new[] { 'a', 'b' }, 9).ToList();
var cas = new (string Nom, Rx Rx, AfN Afn, AfN? Main)[] {
    ("(a|b)*abb", rxAbb, afnAbb, afnQuatre),
    ("(a|b)*aba(a|b)*", rxAba, new Thompson().Construire(rxAba), null),
};
foreach (var (nom, rx, afn, main) in cas) {
    string motif = Rendre(rx);               // l'oracle lit la meme chaine que celle affichee
    var afd = Determiniser(afn);
    var attendu = tous.Select(m => Oracle(motif, m)).ToList();
    var obtenu = tous.Select(m => afn.Accepte(m)).ToList();
    var parAfd = tous.Select(m => AfdAccepte(afd, m)).ToList();
    Console.WriteLine($"{nom,-16} AFN {afn.Etats.Count,2} etats -> AFD {afd.Etats.Count,2} etats ; acceptes {obtenu.Count(o => o),4}/{tous.Count} ; divergences AFN/oracle : {attendu.Zip(obtenu).Count(p => p.First != p.Second)} ; AFN/AFD : {parAfd.Zip(obtenu).Count(p => p.First != p.Second)}");
    if (main is not null) {                  // en plus, l'AFN ecrit a la main dans Automata-01
        var quatre = tous.Select(m => main.Accepte(m)).ToList();
        string libelle = "  (AFN 4 etats)";
        Console.WriteLine($"{libelle,-16} acceptes {quatre.Count(o => o),4}/{tous.Count} ; divergences contre l'oracle : {quatre.Zip(attendu).Count(p => p.First != p.Second)}");
    }
}

// Nombre de classes d'etats equivalents : meme statut final, memes transitions.
// Ce compteur dit si l'automate construit est minimal, sans construire l'automate
// quotient -- c'est le role de Hopcroft dans Automata-01. Le raffinement s'arrete quand
// la partition ne se divise plus.
int ClassesEquivalence(Afd afd) {
    var classe = afd.Etats.ToDictionary(e => e, e => afd.Finals.Contains(e) ? "F" : "N");
    while (true) {
        var signature = afd.Etats.ToDictionary(
            e => e,
            e => classe[e] + ":" + string.Join(",", afd.Alphabet.OrderBy(c => c).Select(c => classe[afd.Delta[(e, c)]])));
        if (signature.Values.Distinct().Count() == classe.Values.Distinct().Count())
            return signature.Values.Distinct().Count();
        classe = signature;
    }
}

Console.WriteLine();
Console.WriteLine("Etats de l'AFD construit, puis classes d'etats equivalents :");
foreach (var (nom, rx) in new (string, Rx)[] {
    ("(a|b)*abb", rxAbb), ("(a|b)*aba(a|b)*", rxAba),
    ("a(b|c)*", rxAbc), ("(ab)*", rxAbEtoile),
}) {
    var afd = Determiniser(new Thompson().Construire(rx));
    Console.WriteLine($"{nom,-16} {afd.Etats.Count,2} etats -> {ClassesEquivalence(afd)} classe(s)");
}

(a|b)*abb        AFN 20 etats -> AFD  5 etats ; acceptes  127/1023 ; divergences AFN/oracle : 0 ; AFN/AFD : 0


  (AFN 4 etats)  acceptes  127/1023 ; divergences contre l'oracle : 0


(a|b)*aba(a|b)*  AFN 30 etats -> AFD  9 etats ; acceptes  560/1023 ; divergences AFN/oracle : 0 ; AFN/AFD : 0


Etats de l'AFD construit, puis classes d'etats equivalents :


(a|b)*abb         5 etats -> 4 classe(s)


(a|b)*aba(a|b)*   9 etats -> 4 classe(s)


a(b|c)*           5 etats -> 3 classe(s)


(ab)*             4 etats -> 3 classe(s)


**Lecture.** Quatre lectures d'un même langage — l'AFN de Thompson, l'AFD obtenu par
sous-ensembles, un `Regex.IsMatch` ancré sur la chaîne rendue, et l'AFN à quatre états
écrit à la main dans `Automata-01` — donnent le même verdict sur chacun des 1023 mots de
longueur au plus 9, sans une divergence.

Le décompte des mots acceptés pour `(a|b)*abb` n'est pas un hasard : sur les 1023 mots,
ceux qui finissent par `abb` sont au nombre de 127 (un mot par préfixe libre, de longueur
0 à 6). Pour `(a|b)*aba(a|b)*`, 560 mots contiennent le facteur `aba`.

Le second tableau est le plus instructif : l'AFD obtenu compte **cinq** états là où le plus
petit en demande quatre, et cinq là où `a(b|c)*` n'en demande que trois. Fusionner les états
équivalents — même statut final, mêmes transitions — ramène 5 à 4, puis 5 à 3 : l'écart
n'est donc pas toujours d'un seul état. La construction par sous-ensembles garantit le
**langage**, jamais la **minimalité** ; c'est le travail de `Hopcroft` dans `Automata-01`,
et c'est aussi pourquoi l'aller-retour du paragraphe 3 revient avec des expressions plus
lourdes que celles d'où il part.


## 3. De l'automate à l'expression : éliminer les états

Le sens inverse est le théorème 3.4, et le manuel en donne **d'abord** une preuve
récursive (§3.2.1 : des expressions $R_{ij}^{(k)}$ qui décrivent les chemins passant par
les états $1..k$), puis une **variante plus économique** au §3.2.2, celle qu'on implémente
ici : *éliminer les états*.

L'idée tient en une phrase, citée du manuel : quand on élimine un état $s$, « les chemins
qui passaient par $s$ n'existent plus dans l'automate ; pour que le langage ne change
pas, il faut ajouter sur un arc allant directement de $q$ à $p$ les étiquettes des chemins
qui allaient de $q$ à $p$ en passant par $s$ ». Comme ces étiquettes sont des ensembles
de mots — parfois infinis — on les écrit comme des expressions régulières, et l'automate
devient un automate **étiqueté par des expressions**.

Pour éliminer $s$, on remplace donc chaque paire (prédécesseur $q_i$, successeur $p_j$)
par l'arc $q_i \to p_j$ étiqueté

$$R(q_i, s)\; R(s, s)^\star\; R(s, p_j)$$

et l'on retire tous les arcs touchant $s$. Quand il ne reste plus que deux nœuds, la
réponse est l'étiquette de l'arc qui les relie. Deux états synthétiques — une source
reliée à l'état initial, un puits relié aux états finaux — ramènent l'automate à un seul
point d'entrée et un seul point de sortie.

Le manuel chiffre le coût des deux méthodes : pour la récursion du §3.2.1, « la longueur
de l'expression peut croître d'un facteur 4 en moyenne à chaque étape d'induction, si bien
que les expressions elles-mêmes peuvent atteindre l'ordre de $4^n$ symboles ». L'élimination
retarde la duplication, elle ne la supprime pas — et **l'ordre dans lequel on élimine les
états change la taille du résultat**, jamais son langage.


In [4]:
#nullable enable

// Union algebrique : le langage vide est neutre, les doublons disparaissent.
Rx? U(Rx? x, Rx? y) {
    if (x is null) return y;
    if (y is null || x == y) return x;
    return new Union(x, y);
}

// Concatenation algebrique : le langage vide est absorbant, epsilon est neutre.
Rx? C(Rx? x, Rx? y) {
    if (x is null || y is null) return null;
    if (x is Eps) return y;
    if (y is Eps) return x;
    return new Concat(x, y);
}

// Etoile algebrique : vide* = epsilon* = epsilon, et (r*)* = r*.
Rx E(Rx? x) {
    if (x is null || x is Eps || x is Etoile) return new Eps();
    return new Etoile(x);
}

// Ordre d'elimination : noms courts d'abord, puis ordre lexicographique.
(int, string) OrdreEtats(string etat) => (etat.Length, etat);

// Elimination d'etats (manuel, section 3.2.2) : arbre d'une expression du langage de l'AFD.
Rx? RegexDepuisAfd(Afd afd, List<string>? ordre = null) {
    var arcs = new Dictionary<(string I, string J), Rx?>();   // (i, j) -> etiquette de l'arc
    // Les arcs sont poses dans un ordre fixe (etat, puis lettre) : meme si deux lettres
    // menent au meme etat, l'ordre de leurs unions ne depend pas de l'ordre d'insertion.
    foreach (var ((p, lettre), q) in afd.Delta.OrderBy(kv => kv.Key.Item1).ThenBy(kv => kv.Key.Item2))
        arcs[(p, q)] = U(arcs.TryGetValue((p, q), out var deja) ? deja : null, new Lit(lettre));
    const string source = "source", puits = "puits";          // etats synthetiques : entree, sortie
    arcs[(source, afd.Initial)] = U(arcs.TryGetValue((source, afd.Initial), out var versInit) ? versInit : null, new Eps());
    foreach (var f in afd.Finals.OrderBy(OrdreEtats))
        arcs[(f, puits)] = U(arcs.TryGetValue((f, puits), out var versPuits) ? versPuits : null, new Eps());
    var vivants = new HashSet<string>(afd.Etats) { source, puits };
    var sequence = ordre ?? afd.Etats.OrderBy(OrdreEtats).ToList();
    foreach (var k in sequence) {
        var boucle = E(arcs.TryGetValue((k, k), out var b) ? b : null);   // R(k, k)*, epsilon si pas de boucle
        var entrants = vivants.Where(i => i != k && arcs.ContainsKey((i, k))).OrderBy(OrdreEtats).ToList();
        var sortants = vivants.Where(j => j != k && arcs.ContainsKey((k, j))).OrderBy(OrdreEtats).ToList();
        foreach (var i in entrants)
            foreach (var j in sortants)
                arcs[(i, j)] = U(arcs.TryGetValue((i, j), out var direct) ? direct : null,
                                 C(C(arcs[(i, k)], boucle), arcs[(k, j)]));
        foreach (var arete in arcs.Keys.Where(a => a.I == k || a.J == k).ToList())
            arcs.Remove(arete);                   // tous les arcs qui touchaient k disparaissent
        vivants.Remove(k);
    }
    return arcs.TryGetValue((source, puits), out var resultat) ? resultat : null;
}

// Premier AFD, ecrit a la main : nombre pair de 1 sur {0, 1} (etat 0 = pair).
var parite = new Afd {
    Etats = new HashSet<string> { "0", "1" },
    Alphabet = new HashSet<char> { '0', '1' },
    Initial = "0",
    Finals = new HashSet<string> { "0" },
};
parite.Delta[("0", '0')] = "0";
parite.Delta[("0", '1')] = "1";
parite.Delta[("1", '0')] = "1";
parite.Delta[("1", '1')] = "0";
var rxParite = RegexDepuisAfd(parite);
Console.WriteLine($"parite du nombre de 1 : '{Rendre(rxParite)}'  ({NbNoeuds(rxParite)} noeuds)");

// Deuxieme AFD : les mots qui contiennent le facteur 01.
var contient01 = new Afd {
    Etats = new HashSet<string> { "0", "1", "2" },
    Alphabet = new HashSet<char> { '0', '1' },
    Initial = "0",
    Finals = new HashSet<string> { "2" },
};
contient01.Delta[("0", '0')] = "1";
contient01.Delta[("0", '1')] = "0";
contient01.Delta[("1", '0')] = "1";
contient01.Delta[("1", '1')] = "2";
contient01.Delta[("2", '0')] = "2";
contient01.Delta[("2", '1')] = "2";
var rx01 = RegexDepuisAfd(contient01);
Console.WriteLine($"contient 01 : '{Rendre(rx01)}'  ({NbNoeuds(rx01)} noeuds)");

// Verification : l'expression produite et l'AFD de depart doivent accepter les memes mots.
var tous01 = Mots(new[] { '0', '1' }, 10).ToList();
foreach (var (nom, afd, rx) in new (string, Afd, Rx?)[] {
    ("parite", parite, rxParite), ("contient 01", contient01, rx01),
}) {
    string motif = Rendre(rx);
    int divergences = tous01.Count(m => AfdAccepte(afd, m) != Oracle(motif, m));
    Console.WriteLine($"{nom,-12} {tous01.Count} mots jusqu'a la longueur 10 : {divergences} divergence(s) contre l'oracle Regex");
}

parite du nombre de 1 : '0*|0*1(0|10*1)*10*'  (22 noeuds)


contient 01 : '1*00*1(0|1)*'  (14 noeuds)


parite       2047 mots jusqu'a la longueur 10 : 0 divergence(s) contre l'oracle Regex


contient 01  2047 mots jusqu'a la longueur 10 : 0 divergence(s) contre l'oracle Regex


**Lecture.** L'élimination redonne `0*|0*1(0|10*1)*10*` pour la parité du nombre de `1`
(22 nœuds). L'expression est exacte — aucune divergence sur les 2047 mots de longueur au
plus 10 — et se relit pourtant : ou bien `0*`, aucun `1` du tout ; ou bien un premier `1`,
puis des blocs qui ajoutent chacun zéro ou deux `1` (`0`, ou `10*1`), puis un dernier `1`
et des zéros. Le compte est pair dans les deux cas.

Le second AFD donne `1*00*1(0|1)*` (14 nœuds), qui se lit d'un trait : « des `1`, un `0`
avec éventuellement d'autres `0`, un `1`, puis n'importe quoi » — c'est bien « contient le
facteur `01` ». Douze caractères contre huit pour `(0|1)*01`, qui décrit le même langage.

L'élimination ne cherche pas la plus petite expression. Elle en cherche une **correcte**,
et rien dans le procédé ne garantit que le résultat soit court.


In [5]:
// Aller-retour complet : expression -> AFN (Thompson) -> AFD (sous-ensembles)
// -> expression (elimination). Le theoreme annonce le meme langage au depart et a
// l'arrivee ; on le verifie sur tous les mots jusqu'a la longueur 7, avec l'oracle Regex
// comme arbitre externe.
foreach (var (nom, rx) in new (string, Rx)[] {
    ("(a|b)*abb", rxAbb), ("a(b|c)*", rxAbc), ("(ab)*", rxAbEtoile),
}) {
    var afn = new Thompson().Construire(rx);
    var afd = Determiniser(afn);
    var retour = RegexDepuisAfd(afd);
    string alphabetEssai = new string(Thompson.Lettres(rx).OrderBy(c => c).ToArray());
    var essai = Mots(alphabetEssai.ToCharArray(), 7).ToList();
    var depart = essai.Select(m => afn.Accepte(m)).ToList();
    var arrivee = essai.Select(m => Oracle(Rendre(retour), m)).ToList();
    Console.WriteLine($"{nom,-10} AFD {afd.Etats.Count,2} etats ; retour '{Rendre(retour)}' ({NbNoeuds(retour)} noeuds) ; {essai.Count} mots, {depart.Zip(arrivee).Count(p => p.First != p.Second)} divergence(s)");
}

(a|b)*abb  AFD  5 etats ; retour '(a|bb*a)(a|b(a|b(a|bb*a)))*bb' (30 noeuds) ; 255 mots, 0 divergence(s)


a(b|c)*    AFD  5 etats ; retour 'a|abb*|(ac|abb*c)(c|bb*c)*(|bb*)' (38 noeuds) ; 3280 mots, 0 divergence(s)


(ab)*      AFD  4 etats ; retour '|ab(ab)*' (10 noeuds) ; 255 mots, 0 divergence(s)


**Lecture.** Trois allers-retours, aucune divergence : la composition des deux théorèmes
rend bien une expression du **même langage**, jamais la même chaîne.

- `(a|b)*abb` : 10 nœuds au départ, 30 à l'arrivée, 255 mots vérifiés ;
- `a(b|c)*` : 6 nœuds au départ, 38 à l'arrivée, 3280 mots vérifiés ;
- `(ab)*` : 4 nœuds au départ, 10 à l'arrivée, 255 mots vérifiés.

Les trois reviennent plus lourdes, et le passage par un AFD non minimal y est pour
quelque chose : chaque état de trop est un état de plus où l'élimination duplique des
étiquettes. Le dernier cas montre aussi comment $\varepsilon$ s'écrit dans ce rendu : une
alternative vide en tête, `|ab(ab)*`, que le moteur de regex lit comme « le mot vide, ou
`ab` répété ».


## 4. Ce que coûte la conversion

Les trois conversions n'ont pas le même régime de coût, et c'est ce qui décide de leur
usage :

| Conversion | Majorant | Pourquoi |
|---|---|---|
| expression → AFN | **linéaire** : deux états par nœud | chaque nœud ajoute deux états, jamais plus |
| AFN → AFD | jusqu'à $2^n$ pour $n$ états | un état de l'AFD est un *ensemble* d'états de l'AFN |
| AFD → expression | peut exploser (le manuel cite $4^n$ pour la récursion) | chaque élimination duplique les étiquettes |

Plutôt que de réciter ces majorants, mesurons-les sur deux familles paramétrées. La
première est la famille classique $(a|b)^\star a\,(a|b)^n$, dont l'AFD minimal a
$2^{n+1}$ états. La seconde est la famille « le mot finit par $a^n$ », dont l'AFD a
$n+1$ états : on lui applique l'élimination dans deux ordres différents.


In [6]:
#nullable enable

// AFD a n+1 etats : les mots sur {a, b} dont le suffixe est a^n.
Afd AfdFinitParA(int n) {
    var afd = new Afd {
        Alphabet = new HashSet<char> { 'a', 'b' },
        Initial = "0",
        Finals = new HashSet<string> { n.ToString() },
    };
    for (int i = 0; i <= n; i++) {
        afd.Etats.Add(i.ToString());
        afd.Delta[(i.ToString(), 'a')] = Math.Min(i + 1, n).ToString();   // on avance, puis on sature
        afd.Delta[(i.ToString(), 'b')] = "0";                             // b casse la serie
    }
    return afd;
}

Console.WriteLine("Determinisation : (a|b)*a(a|b)^n, dont l'AFD minimal a 2^(n+1) etats");
string h0 = "n", h1 = "etats AFN", h2 = "etats AFD", h3 = "apres fusion";
Console.WriteLine($"{h0,2} {h1,10} {h2,10} {h3,13}");
for (int n = 1; n <= 8; n++) {
    Rx rx = new Concat(new Etoile(AB), new Lit('a'));
    for (int i = 0; i < n; i++) rx = new Concat(rx, AB);
    var afn = new Thompson().Construire(rx);
    var afd = Determiniser(afn);
    Console.WriteLine($"{n,2} {afn.Etats.Count,10} {afd.Etats.Count,10} {ClassesEquivalence(afd),13}");
}

Console.WriteLine();
Console.WriteLine("Elimination : famille 'suffixe a^n' (n+1 etats), ordre naturel puis ordre inverse");
string g0 = "n", g1 = "noeuds", g2 = "caracteres", g3 = "noeuds inv.", g4 = "car. inv.";
Console.WriteLine($"{g0,2} {g1,7} {g2,11} {g3,12} {g4,10}");
for (int n = 1; n <= 6; n++) {
    var afd = AfdFinitParA(n);
    var naturel = RegexDepuisAfd(afd);
    var inverse = RegexDepuisAfd(afd, afd.Etats.OrderBy(OrdreEtats).Reverse().ToList());
    Console.WriteLine($"{n,2} {NbNoeuds(naturel),7} {Rendre(naturel).Length,11} {NbNoeuds(inverse),12} {Rendre(inverse).Length,10}");
    if (n == 4) {
        // Les deux ordres doivent donner le meme langage : on le verifie au passage.
        var essai = Mots(new[] { 'a', 'b' }, 8).ToList();
        int ecart = essai.Count(m => Oracle(Rendre(naturel), m) != Oracle(Rendre(inverse), m));
        Console.WriteLine($"   n=4 : les deux ordres divergent sur {ecart} des {essai.Count} mots testes");
    }
}

Determinisation : (a|b)*a(a|b)^n, dont l'AFD minimal a 2^(n+1) etats


 n  etats AFN  etats AFD  apres fusion


 1         20          5             4


 2         28          9             8


 3         36         17            16


 4         44         33            32


... (9 outputs left out)

 4     154         124           32         30


   n=4 : les deux ordres divergent sur 0 des 511 mots testes


 5     314         252           38         36


 6     634         508           44         42


**Lecture.** La déterminisation **double** d'un pas à l'autre : 5, 9, 17, 33, …, 513
états pour l'AFD, quand l'AFN ne gagne que huit états par pas (20, 28, 36, …, 76). Ce
doublement est l'exponentielle du tableau, mesurée et non récitée. La dernière colonne
montre aussi que l'automate construit n'est pas minimal : une fois les états équivalents
fusionnés, il reste exactement $2^{n+1}$ états — **un** état de trop à chaque pas, jamais
davantage.

Sur la seconde famille, les deux ordres d'élimination ne coûtent pas du tout la même
chose. Dans l'ordre **inverse** — l'état final d'abord — la taille croît **linéairement** :
14, 20, 26, 32, 38, 44 nœuds, six de plus par pas. Dans l'ordre **naturel** — l'état
initial d'abord — elle double à peu près à chaque pas : 14, 34, 74, 154, 314, 634 nœuds.
Les deux expressions décrivent pourtant le même langage : vérifié sur 511 mots au cas
$n = 4$, zéro divergence.

L'explication tient à ce que l'élimination transporte. Dans cet AFD, **toutes** les flèches
étiquetées `b` reviennent à l'état 0 : éliminer 0 en premier oblige à recopier sa boucle
dans les arcs qui partent des autres états, et chaque élimination suivante duplique à
nouveau ces étiquettes. Éliminer l'état final en premier fait au contraire replier la
chaîne des `a` sur un seul arc, dont l'étiquette gagne une lettre par pas — d'où la
croissance linéaire.

À retenir pour la conception : écrire une expression coûte peu, la déterminiser peut coûter
cher, et revenir à une expression depuis un automate coûte ce que coûte l'ordre choisi.


## 5. Ce que le dépôt fait déjà avec les expressions régulières

Trois carnets voisins traitent la même matière par d'autres routes. Les renvois sont
précis, pour que le lecteur sache quoi y chercher.

- **`SymbolicAI/SMT/Z3-API/Z3-04-Strings-Regex-*`**, sections 3 à 5 : les expressions
  régulières y sont des **objets symboliques** du solveur (`Re`, `InRe`). La différence
  avec ce carnet est celle de l'usage : ici l'expression est un **programme** qu'on exécute,
  là-bas c'est une **contrainte** que le solveur décide — et dont il peut exhiber un mot
  témoin. C'est le pont naturel vers les questions de la forme « existe-t-il un mot qui
  satisfait ces deux motifs à la fois ? ».
- **`Search/Part1-Foundations/Search-10-SymbolicAutomata`**, sections 3 et 4 : les
  transitions y sont étiquetées par des **prédicats** Z3 (intervalles d'entiers), ce qui
  autorise un alphabet infini. La construction de Thompson suppose un alphabet fini et
  énumérable : c'est exactement l'hypothèse que les automates symboliques relâchent.
- **`SymbolicAI/Lean/Lean-14-Finiteness-Derivatives`** : les **dérivées de Brzozowski**
  construisent l'automate *directement depuis l'arbre*, sans passer par les
  epsilon-transitions. Le carnet formalise la finitude de l'ensemble des dérivées, qui est
  ce qui garantit un matching linéaire. C'est la troisième route entre expression et
  automate, après Thompson (section 2) et l'élimination (section 3).

## Bilan

- Une expression régulière se convertit en automate par **induction structurelle** : trois
  cas de base, trois cas de composition, deux états par nœud, et des epsilon-transitions
  qui viennent de la construction elle-même.
- Un automate se convertit en expression par **élimination d'états** : la boucle d'un état
  devient une étoile, chaque paire (prédécesseur, successeur) reçoit l'étiquette du détour,
  et l'ordre d'élimination arbitre entre taille et lisibilité.
- Les deux sens se composent, et l'aller-retour revient sur le **même langage** — vérifié
  mot par mot contre un moteur indépendant.
- Le coût n'est pas symétrique : linéaire dans un sens, potentiellement exponentiel dans
  l'autre. Une expression courte peut cacher un automate énorme.

Ce que ce carnet ne fait pas : décider si un langage **est** régulier. C'est l'objet du
lemme de pompage, déjà abordé en exercice dans `Automata-01`, et le point de départ de la
sous-série : les automates à pile (`Automata-03`) quittent la classe des langages réguliers.


## Exercices

Trois exercices, dans l'ordre du carnet. Chaque cellule s'exécute **sans erreur telle
qu'elle est livrée** : remplacez les lignes `# TODO etudiant` par votre solution, puis
relancez la cellule.


### Exercice 1 — Construire l'automate d'une expression

Construisez l'arbre de l'expression $(ab \mid c)^\star a$ avec les constructeurs du
paragraphe 1 (`new Lit('a')`, `new Union(...)`, `new Concat(...)`, `new Etoile(...)`), puis
laissez la cellule vérifier le langage obtenu contre le motif `(ab|c)*a` sur tous les mots
de longueur au plus 7 sur $\{a, b, c\}$.

*(Indice : `ab` est une concaténation de deux littéraux, `(ab|c)` une union, et l'étoile
porte sur cette union avant la concaténation finale avec `a`.)*


In [7]:
#nullable enable

// TODO etudiant : construire l'arbre de (ab|c)*a (remplacez null).
Rx? rxEx1 = null;

// Verification : a executer apres votre construction, sans la modifier.
if (rxEx1 is not null) {
    var afnEx1 = new Thompson().Construire(rxEx1);
    var essai = Mots(new[] { 'a', 'b', 'c' }, 7).ToList();
    int divergences = essai.Count(m => afnEx1.Accepte(m) != Oracle("(ab|c)*a", m));
    Console.WriteLine($"{essai.Count} mots testes ; divergences : {divergences} (attendu 0)");
} else {
    Console.WriteLine("Exercice a completer : construisez rxEx1 (un arbre) puis relancez cette cellule.");
}

Exercice a completer : construisez rxEx1 (un arbre) puis relancez cette cellule.


### Exercice 2 — Éliminer les états à la main

Voici un AFD à trois états sur $\{0, 1\}$ qui accepte les mots se terminant par `01` :

| État | sur `0` | sur `1` |
|---|---|---|
| 0 (initial) | 1 | 0 |
| 1 | 1 | 2 |
| 2 (final) | 1 | 0 |

Appliquez l'élimination d'états **sur papier**, puis écrivez le résultat comme une chaîne
C# lisible par `Regex`. La cellule vérifie votre chaîne contre l'AFD sur tous les mots de
longueur au plus 8.


In [8]:
#nullable enable

var afdEx2 = new Afd {
    Etats = new HashSet<string> { "0", "1", "2" },
    Alphabet = new HashSet<char> { '0', '1' },
    Initial = "0",
    Finals = new HashSet<string> { "2" },
};
afdEx2.Delta[("0", '0')] = "1";
afdEx2.Delta[("0", '1')] = "0";
afdEx2.Delta[("1", '0')] = "1";
afdEx2.Delta[("1", '1')] = "2";
afdEx2.Delta[("2", '0')] = "1";
afdEx2.Delta[("2", '1')] = "0";

// TODO etudiant : ecrire ici l'expression trouvee a la main (chaine C#, remplacez null).
string? chaineEx2 = null;

// Verification : a executer apres votre derivation, sans la modifier.
if (chaineEx2 is not null) {
    var essai = Mots(new[] { '0', '1' }, 8).ToList();
    int divergences = essai.Count(m => AfdAccepte(afdEx2, m) != Oracle(chaineEx2, m));
    Console.WriteLine($"{essai.Count} mots testes ; divergences : {divergences} (attendu 0)");
    Console.WriteLine($"Votre expression : '{chaineEx2}' ; celle de l'organe du paragraphe 3 : '{Rendre(RegexDepuisAfd(afdEx2))}'");
} else {
    Console.WriteLine("Exercice a completer : chaineEx2 = \"...\";  // votre expression");
    Console.WriteLine("Indication : l'oracle exige que l'expression decrive le mot ENTIER,");
    Console.WriteLine("du premier caractere au dernier ; \"01\" seul ne reconnait que le mot 01.");
}

Exercice a completer : chaineEx2 = "...";  // votre expression


Indication : l'oracle exige que l'expression decrive le mot ENTIER,


du premier caractere au dernier ; "01" seul ne reconnait que le mot 01.


### Exercice 3 — Pourquoi l'ordre d'élimination compte

Dans le tableau du paragraphe 4, la famille « suffixe $a^n$ » donne une expression dont
la taille croît **linéairement** quand on élimine les états du dernier au premier, et
**double à peu près à chaque pas** quand on les élimine du premier au dernier.

**Expliquez en trois phrases** d'où vient cette différence, en vous appuyant sur la règle
d'élimination $R(q_i, s)\,R(s, s)^\star\,R(s, p_j)$ : quels arcs sont dupliqués, et
pourquoi l'ordre décide de la taille des étiquettes qu'on transporte. *(Indice : dans cet
AFD, toutes les flèches étiquetées `b` reviennent à l'état 0. Regardez ce qui se recopie
sur les autres arcs quand on élimine 0 en premier, et ce qui se passe au contraire quand
on élimine l'état final en premier.)*


In [9]:
// Reponse a rediger en commentaire (exercice de raisonnement).
// TODO etudiant : quand on elimine l'etat ...
// ... donc l'expression naturelle ...
Console.WriteLine("Exercice de raisonnement : redigez votre reponse en commentaire ci-dessus.");

Exercice de raisonnement : redigez votre reponse en commentaire ci-dessus.
